# ReVoice — Voice Cloning API (XTTS-v2 + F5-TTS)

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save

Run cells **top to bottom, once each**. Don't run any old cells from previous notebooks.

Licence note: F5-TTS model weights are released for **non-commercial use** (fine for a college project).

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Install everything (3–5 minutes)
Installs XTTS, then F5-TTS **without letting it change other packages**, then ONE final line pins the known-working versions, then a conflict check (diffusers / gradio warnings can be ignored: ReVoice doesn't use them).
**After this finishes: Runtime → Restart session**, then continue from Step 2b.

In [ ]:
# XTTS + server tools
!pip install -q coqui-tts fastapi uvicorn pyngrok soundfile

# F5-TTS: --no-deps so it can't overwrite transformers/numpy
!pip install -q --no-deps f5-tts cached_path
!pip install -q ema_pytorch torchdiffeq vocos x_transformers hydra-core omegaconf jieba pypinyin unidecode pydub wandb
!pip install -q rjieba || echo "rjieba optional - skipped"

# ffmpeg converts the browser's webm/opus recordings
!apt-get -qq install -y ffmpeg > /dev/null

# ONE final pinning line with the known-working versions.
# It must stay LAST so no earlier install can change these afterwards.
!pip install -q tomli transformers_stream_generator boto3 bitsandbytes "transformers==4.57.6" "huggingface-hub>=0.34.0,<1.0" "tokenizers>=0.22.0,<=0.23.0" "numpy<2.3"

# Conflict check, only for the packages ReVoice actually uses
# (diffusers / gradio complaints are ignored: we don't use them)
!pip check 2>&1 | grep -iE "transformers|coqui|tts|f5|librosa|numba|torch|cached" | grep -viE "^(diffusers|gradio)" || echo "No conflicts in the packages ReVoice uses."

print("Install finished. Now: Runtime -> Restart session, then run Step 2b.")

## Step 2b — Check versions (after restarting)
If this errors, don't continue.

In [ ]:
import transformers, numpy, librosa, tokenizers, huggingface_hub
from packaging.version import Version
from transformers.pytorch_utils import isin_mps_friendly
from transformers.utils.import_utils import is_torch_greater_or_equal
print("transformers   ", transformers.__version__)
print("tokenizers     ", tokenizers.__version__)
print("huggingface-hub", huggingface_hub.__version__)
print("numpy          ", numpy.__version__)

# compare with the known-working set from Step 2
problems = []
if transformers.__version__ != "4.57.6":
    problems.append(f"transformers is {transformers.__version__}, expected 4.57.6")
if not Version("0.22.0") <= Version(tokenizers.__version__) <= Version("0.23.0"):
    problems.append(f"tokenizers is {tokenizers.__version__}, expected 0.22.0 to 0.23.0")
if not Version("0.34.0") <= Version(huggingface_hub.__version__) < Version("1.0"):
    problems.append(f"huggingface-hub is {huggingface_hub.__version__}, expected 0.34 up to (not incl.) 1.0")
if Version(numpy.__version__) >= Version("2.3"):
    problems.append(f"numpy is {numpy.__version__}, expected below 2.3")

import f5_tts.api
print("f5-tts importable")
if problems:
    raise RuntimeError("Wrong versions:\n - " + "\n - ".join(problems) +
                       "\nRe-run Step 2, then Runtime -> Restart session.")
print("All checks passed")

## Step 3 — Load XTTS-v2 (~2 GB download first time)

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
xtts_model = tts.synthesizer.tts_model
SAMPLE_RATE = 24000
print("XTTS-v2 loaded on", device)

## Step 3b — Load F5-TTS (~1.5 GB download first time)
Also warms up Whisper (the speech-to-text model F5 uses for the reference transcript), so it downloads now and not during your first request.
If this fails, XTTS still works — the app will report the F5 error when you pick it.

In [ ]:
f5 = None
try:
    from f5_tts.api import F5TTS
    f5 = F5TTS(device=device)
    print("F5-TTS loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nF5-TTS failed to load (XTTS still available):", e)

if f5 is not None:
    # Warm-up: the first transcription downloads Whisper (~1.5 GB).
    # Doing it now means the first real request isn't slow or timing out.
    try:
        from importlib.resources import files
        example = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
        print("Whisper warm-up transcript:", f5.transcribe(example))
    except Exception as e:
        print("Whisper warm-up failed (F5 will try again on the first request):", e)

## Step 4 — Quick test of both models

In [ ]:
from IPython.display import Audio, display
from importlib.resources import files

print("XTTS-v2:")
display(Audio(tts.tts(text="Hello, this is the XTTS voice.", speaker="Ana Florence", language="en"), rate=SAMPLE_RATE))

if f5 is not None:
    print("F5-TTS:")
    ref = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
    wav, sr, _ = f5.infer(ref_file=ref,
                          ref_text="Some call me nature, others call me mother nature.",
                          gen_text="Hello, this is the F5 voice.")
    display(Audio(wav, rate=sr))

## Step 4a — Load the ReVoice voice functions
The code that turns text + reference recordings into speech. The API in Step 6 and the test in Step 4b both use these, so 4b tests exactly what the app gets.

In [ ]:
# The functions that turn text + reference recordings into speech.
# They live HERE (not inside Step 6) so that Step 4b tests EXACTLY the
# same code the API uses.
import os, base64, hashlib, subprocess, tempfile
import soundfile as sf
from types import SimpleNamespace
try:
    import f5_tts.infer.utils_infer as f5_utils     # F5's own reference trimming + caches
except Exception:
    f5_utils = None                                 # F5 not installed: XTTS still works

xtts_cache = {}   # voice profiles for XTTS
f5_cache = {}     # (trimmed reference clip, its transcript) for F5
temp_files = []   # cleaned clips on disk, deleted by forget_all_voices()

# XTTS: emotion changes speaking speed and how "lively" it sounds
EMOTION_STYLE = {
    "neutral": {"speed": 1.00, "temperature": 0.80},
    "happy":   {"speed": 1.05, "temperature": 0.85},
    "urgent":  {"speed": 1.12, "temperature": 0.85},
    "tender":  {"speed": 0.92, "temperature": 0.75},
}

# F5 settings
F5_SPEED = 1.0              # always normal speed: the reference recording carries the emotion
F5_NFE_STEPS = 64           # more refinement steps = clearer speech (32 was a bit rough)
F5_MAX_REF_SECONDS = 15     # pass up to 15s; F5 then trims to <=12s at a pause by itself

def clean_clip(b64, max_seconds=None):
    """Decode a browser recording and clean it for cloning."""
    raw = base64.b64decode(b64)
    f = tempfile.NamedTemporaryFile(suffix=".input", delete=False)
    f.write(raw); f.close()
    out = f.name + ".wav"
    cmd = ["ffmpeg", "-y", "-i", f.name, "-ac", "1", "-ar", "24000",
           "-af", "highpass=f=70,"
                  "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,"
                  "loudnorm=I=-18:TP=-2:LRA=11"]
    if max_seconds:
        cmd += ["-t", str(max_seconds)]
    r = subprocess.run(cmd + [out], capture_output=True, text=True)
    os.remove(f.name)              # the raw upload isn't needed any more
    if r.returncode != 0:
        raise RuntimeError("ffmpeg failed:\n" + r.stderr[-1200:])
    temp_files.append(out)
    return out

def usable_clips(clips_b64, max_seconds=None):
    out = []
    for b64 in clips_b64:
        p = clean_clip(b64, max_seconds)
        d = sf.info(p).duration
        if d >= 0.8:
            out.append((p, d))
    if not out:
        raise RuntimeError("All reference clips were too short or silent. Re-record and keep talking.")
    return out

def clips_key(clips_b64):
    return hashlib.md5("".join(clips_b64).encode()).hexdigest()

# ---------- XTTS (cloned from the user's recordings) ----------
def run_xtts(clips_b64, req, style):
    key = clips_key(clips_b64)
    if key not in xtts_cache:
        clips = usable_clips(clips_b64)
        print(f"   XTTS profile from {sum(d for _, d in clips):.1f}s of audio")
        xtts_cache[key] = xtts_model.get_conditioning_latents(
            audio_path=[p for p, _ in clips], gpt_cond_len=30, gpt_cond_chunk_len=4, max_ref_length=30)
    gpt_cond_latent, speaker_embedding = xtts_cache[key]
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# ---------- XTTS with a stock (built-in) voice ----------
def stock_speaker_names():
    return sorted(xtts_model.speaker_manager.speaker_names)

def run_xtts_stock(req, style):
    speakers = xtts_model.speaker_manager.speakers
    if req.stock_speaker not in speakers:
        raise RuntimeError(f"Unknown stock voice {req.stock_speaker!r}. "
                           f"Some valid ones: {', '.join(stock_speaker_names()[:8])}")
    # built-in voices come with their voice profile already computed
    gpt_cond_latent, speaker_embedding = speakers[req.stock_speaker].values()
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# ---------- F5-TTS ----------
def run_f5(clips_b64, req):
    if f5 is None:
        raise RuntimeError("F5-TTS isn't loaded. Check the output of Step 3b.")
    key = clips_key(clips_b64)
    if key not in f5_cache:
        # F5 uses ONE reference: the longest clip, up to 15 seconds
        clips = usable_clips(clips_b64, max_seconds=F5_MAX_REF_SECONDS)
        longest = max(clips, key=lambda c: c[1])[0]
        # Let F5 trim the clip FIRST (to <=12s, at a pause), and give it an empty
        # transcript so Whisper writes down the TRIMMED audio. The transcript then
        # matches exactly the words F5 hears; a mismatch causes mumbled/skipped words.
        ref_path, ref_text = f5_utils.preprocess_ref_audio_text(longest, "", show_info=lambda *a, **k: None)
        print(f"   F5 reference: {sf.info(ref_path).duration:.1f}s, transcript: {ref_text!r}")
        f5_cache[key] = (ref_path, ref_text)
    ref_path, ref_text = f5_cache[key]
    wav, sr, _ = f5.infer(ref_file=ref_path, ref_text=ref_text, gen_text=req.text,
                          speed=F5_SPEED, nfe_step=F5_NFE_STEPS, show_info=lambda *a, **k: None)
    return wav, sr, {"ref_text": ref_text.strip()}

# ---------- one entry point, used by the API and by Step 4b ----------
def synthesize_core(req):
    """Returns (wav, sample_rate, extra_info). req needs: text, language, emotion,
    engine, stock_speaker, speaker_wav_base64, speaker_wav_base64_list."""
    style = EMOTION_STYLE.get(req.emotion, EMOTION_STYLE["neutral"])
    if req.stock_speaker:
        # stock voice: no recordings needed (XTTS only)
        if req.engine != "xtts":
            raise RuntimeError("Stock voices only work with XTTS-v2.")
        print(f"\n--- [xtts stock] {req.text!r} | {req.emotion} | {req.stock_speaker}")
        return run_xtts_stock(req, style)

    clips = req.speaker_wav_base64_list or ([req.speaker_wav_base64] if req.speaker_wav_base64 else [])
    if not clips:
        raise RuntimeError("No reference audio sent.")
    print(f"\n--- [{req.engine}] {req.text!r} | {req.emotion} | {len(clips)} clip(s)")
    if req.engine == "f5":
        return run_f5(clips, req)
    if req.engine == "xtts":
        return run_xtts(clips, req, style)
    raise RuntimeError(f"Unknown engine {req.engine!r}.")

def forget_all_voices():
    """Wipes every cached voice and cleaned clip (called when a user is deleted)."""
    n = len(xtts_cache) + len(f5_cache)
    xtts_cache.clear(); f5_cache.clear()
    # F5 keeps its own cache of trimmed clips and transcripts: wipe that too
    if f5_utils is not None:
        temp_files.extend(f5_utils._ref_audio_cache.values())
        f5_utils._ref_audio_cache.clear(); f5_utils._ref_text_cache.clear()
    for path in temp_files:
        try:
            os.remove(path)
        except OSError:
            pass
    temp_files.clear()
    print(f"\n--- forgot {n} cached voice(s)")
    return n

print("Voice functions ready.")

## Step 4b — Test with YOUR voice (optional, for troubleshooting)
Upload a short recording (6–20 s). XTTS (and F5) speak a fixed sentence through the exact API code.
- Gibberish **here too** → environment problem: check Step 2b, redo Step 2.
- Fine here but bad in the app → the reference clips sent by the app are the problem (re-record).

In [ ]:
# Upload one recording of the voice you're testing (wav, mp3, m4a, webm, ogg...).
# XTTS then says a fixed sentence using EXACTLY the same code as the API.
#   Gibberish HERE too        -> environment problem: redo Step 2 (check Step 2b)
#   Fine here, bad in the app -> problem with the reference clips the app sent
import time
from google.colab import files as colab_files
from IPython.display import Audio, display

TEST_SENTENCE = "Hello, this is a quick test of my cloned voice. Does it sound like me?"
ALSO_TEST_F5 = True     # set to False to test XTTS only

uploaded = colab_files.upload()
for name, data in uploaded.items():
    req = SimpleNamespace(text=TEST_SENTENCE, language="en", emotion="neutral", engine="xtts",
                          stock_speaker=None, speaker_wav_base64=None,
                          speaker_wav_base64_list=[base64.b64encode(data).decode()])
    t0 = time.time()
    wav, sr, _ = synthesize_core(req)
    print(f"{name} -> XTTS ({time.time() - t0:.1f}s):")
    display(Audio(wav, rate=sr))

    if ALSO_TEST_F5 and f5 is not None:
        req.engine = "f5"
        t0 = time.time()
        wav, sr, extra = synthesize_core(req)
        print(f"{name} -> F5 ({time.time() - t0:.1f}s). Whisper heard the reference as: {extra['ref_text']!r}")
        display(Audio(wav, rate=sr))

## Step 5 — ngrok token
https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

## Step 6 — Start the API (both engines)
Safe to re-run any time. Your ngrok URL stays the same.
The last line must say **ReVoice API v6**.
Needs Step 4a to have been run (the API uses the same voice functions as Step 4b).

In [ ]:
import traceback, threading, time, requests, socket, io
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
from pyngrok import ngrok

# stop a server/tunnel left over from running this cell before
try:
    server.should_exit = True; time.sleep(2)
except NameError:
    pass
try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = FastAPI()

class SynthRequest(BaseModel):
    text: str
    speaker_wav_base64: Optional[str] = None
    speaker_wav_base64_list: Optional[List[str]] = None
    language: str = "en"
    emotion: str = "neutral"
    engine: str = "xtts"
    stock_speaker: Optional[str] = None   # built-in XTTS voice, for users with no recordings

@app.get("/")
def health():
    return {"status": "ReVoice API v6 running", "xtts": True, "f5": f5 is not None}

@app.get("/speakers")
def speakers():
    """Names of XTTS's built-in voices (for the 'use a stock voice' option)."""
    return {"speakers": stock_speaker_names()}

@app.post("/forget_voices")
def forget_voices():
    """Wipes every cached voice profile and cleaned clip (called when a user is deleted)."""
    return {"forgotten": forget_all_voices()}

@app.post("/synthesize")
def synthesize(req: SynthRequest):
    try:
        t0 = time.time()
        wav, sr, extra = synthesize_core(req)        # defined in Step 4a
        print(f"   generated in {time.time() - t0:.1f}s")
        buf = io.BytesIO()
        sf.write(buf, wav, samplerate=sr, format="WAV")
        # extra = {"ref_text": ...} for F5: the words Whisper heard in the reference
        return {"audio_base64": base64.b64encode(buf.getvalue()).decode("utf-8"), **extra}
    except Exception as e:
        print(traceback.format_exc())
        return JSONResponse({"error": f"{type(e).__name__}: {e}"}, status_code=500)

def free_port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p

PORT = free_port()
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

# wait (up to 30s) until the server is really listening, instead of guessing
for _ in range(60):
    if server.started:
        break
    time.sleep(0.5)
else:
    raise RuntimeError("The API server did not start within 30 seconds. "
                       "Look for an error message above, then re-run this cell.")

tunnel = ngrok.connect(PORT)
print("=" * 65)
print("URL for COLAB_API_URL in backend/.env:")
print(tunnel.public_url)
print("=" * 65)

# health check through ngrok; show the raw reply if it isn't JSON (e.g. an ngrok error page)
reply = requests.get(tunnel.public_url, headers={"ngrok-skip-browser-warning": "true"}, timeout=30)
try:
    print(reply.json())
except ValueError:
    print(f"Health check did not return JSON (HTTP {reply.status_code}). Raw reply:")
    print(reply.text[:500])

## Keep this tab open
If Colab disconnects: run Steps 3, 3b, 4a, 5 and 6 again (Step 2 only if packages are missing).
Never run cells from older notebooks — the URL always points to whichever server started last.